# Measuring Gendered Interruptions at the Supreme Court of Canada
## A Computational Analysis of Judicial Power Dynamics in Oral Argument

**Group Members:**
- Sabrin Saide (221178603)
- Matt Aydin (220185328)
- Gobind Dhugee (221173794)

**Course:** Legal Tech Coding — Professor Rehaag, Winter 2026

---

**Research Question:** Can AI-generated transcripts be used to measure whether gendered interruption patterns exist during Supreme Court of Canada oral hearings?


## 1. Setup

First, we clone our project repository from GitHub and install the required Python packages. This only needs to run once.

In [ ]:
import os
import subprocess
import sys

# Clone the repo if we haven't already
if not os.path.exists("scc-interruptions"):
    subprocess.run(["git", "clone", "https://github.com/matt-aydin-2000/scc-interruptions.git"], check=True)
    print("Repository cloned.")
else:
    print("Repository already exists.")

# Install dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "scc-interruptions/requirements.txt"], check=True)
print("Dependencies installed.")


## 2. Import Our Modules

We add the project directory to Python's path so we can import our custom modules.

In [ ]:
import sys
sys.path.insert(0, "scc-interruptions")
os.chdir("scc-interruptions")

from scraper import fetch_all_transcripts
from transcript_parser import parse_transcript_file
from interruption_detector import detect_interruptions, compute_interruption_metrics, compute_time_to_first_interruption
from analysis import build_justice_dataframe, build_case_level_dataframe, full_analysis, format_results_report
from visualizations import generate_all_visualizations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
%matplotlib inline

print("All modules imported successfully.")


## 3. Data Acquisition

We scrape 121 AI-generated SCC hearing transcripts from [obiter.ai](https://obiter.ai/scc/), Simon Wallace's project. Wallace transcribed these hearings using OpenAI's Whisper (speech-to-text) and Pyannote (speaker diarization). See: Wallace, S. (2023), "Speaking Like a Judge," *Supreme Court Law Review*, 115(1).

The scraper caches downloaded files, so re-running this cell won't re-download transcripts you already have.

In [ ]:
RAW_DATA_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
OUTPUT_DIR = "output"

os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

fetch_all_transcripts(RAW_DATA_DIR, delay=1.0)


## 4. Parse Transcripts into Speaker Turns

Each transcript is HTML with a pattern like `Justice Kasirer (00:09:02): Vous n'allez pas...`. Our parser extracts the speaker name, timestamp, and text from each turn, then tags whether the speaker is a justice or counsel. Justice gender is coded from official SCC pronouns, following Wallace's methodology.

In [ ]:
raw_files = sorted([f for f in os.listdir(RAW_DATA_DIR) if f.endswith(".json")])

cases = []
for filename in raw_files:
    filepath = os.path.join(RAW_DATA_DIR, filename)
    try:
        case = parse_transcript_file(filepath)
        if case["turns"]:
            cases.append(case)
    except Exception as e:
        print(f"  [ERROR] {filename}: {e}")

total_turns = sum(c["total_turns"] for c in cases)
print(f"Parsed {len(cases)} cases with {total_turns} total speaker turns.")

# Show a sample turn so you can see the data structure
print("\nSample speaker turn:")
for k, v in cases[0]["turns"][0].items():
    print(f"  {k}: {v}")


## 5. Detect Interruptions

We detect interruptions using three methods adapted from the US Supreme Court literature (Jacobi & Schweers 2017, Feldman & Gill 2019):

1. **Overlap** — the diarization model explicitly flagged overlapping speech
2. **Timing-based** — a new speaker starts within 15 seconds AND the previous speaker's text looks cut off (ends with a dash, comma, conjunction, etc.)
3. **Rapid judicial intervention** — a justice speaks before counsel has said more than 50 words

We use a 15-second threshold, calibrated through pilot testing.

In [ ]:
all_interruptions = []
all_metrics = []
all_case_data = []
all_ttfi = []

for case in cases:
    turns = case["turns"]
    interruptions = detect_interruptions(turns, time_threshold=15)
    all_interruptions.extend(interruptions)

    for intr in interruptions:
        intr["case_id"] = case["case_id"]
        intr["case_date"] = case["date"]

    metrics = compute_interruption_metrics(interruptions, turns)
    all_metrics.append((case["case_id"], metrics))
    all_case_data.append((case["case_id"], case["date"], metrics, case))

    ttfi = compute_time_to_first_interruption(turns, interruptions)
    all_ttfi.extend(ttfi)

n_overlap = sum(1 for i in all_interruptions if i["type"] == "overlap")
n_timing = sum(1 for i in all_interruptions if i["type"] == "timing")
n_rapid = sum(1 for i in all_interruptions if i["type"] == "rapid_intervention")

print(f"Total interruptions found: {len(all_interruptions)}")
print(f"  Overlapping speech:  {n_overlap}")
print(f"  Timing-based:        {n_timing}")
print(f"  Rapid intervention:  {n_rapid}")


## 6. Justice-Level Summary

We aggregate interruption counts across all cases for each justice, normalized per 1,000 words spoken to control for volubility (Feldman & Gill 2019 found that speaking an additional 1,000 words increases interruption rate by ~40%).

In [ ]:
justice_df = build_justice_dataframe(all_metrics)
case_df = build_case_level_dataframe(all_case_data)

display_cols = ["justice", "gender", "cases_heard", "total_words_spoken",
                "interruptions_made", "interruptions_received",
                "rate_made_per_1k_words", "rate_received_per_1k_words"]
justice_df[display_cols].round(2)


## 7. Statistical Analysis

We run the full battery of tests:
- **Descriptive statistics** (means, medians by gender)
- **Welch's t-tests** and **Mann-Whitney U tests** (comparing male vs female rates)
- **Negative binomial regression** (predicting interruption count from gender, controlling for volubility and Chief Justice status)
- **Cohen's d** effect sizes
- **Z-score outlier analysis**

In [ ]:
results = full_analysis(justice_df, case_df)
report = format_results_report(results)
print(report)


## 8. Visualizations

### Interruption Rates by Gender

In [ ]:
fig_paths = generate_all_visualizations(justice_df, all_interruptions, all_ttfi, OUTPUT_DIR)

from IPython.display import Image, display
for path in fig_paths:
    display(Image(filename=path))


## 9. Key Findings

**Interruptions made:** No statistically significant gender difference. Male justices averaged 2.58 interruptions per 1,000 words spoken; female justices averaged 2.51 (t-test p = 0.92, Mann-Whitney p = 0.48). Cohen's d = 0.057 (negligible effect size).

**Interruptions received:** Male justices appeared to receive more interruptions on average (2.39 vs 1.01 per 1,000 words), but this was driven almost entirely by Chief Justice Wagner (z-score = 2.98), whose procedural role as hearing manager inflates his count. After regression controls, gender was not a significant predictor (p = 0.97).

**Regression:** Neither model (interruptions made or received) found gender to be a significant predictor after controlling for volubility and Chief Justice status.

**Outlier:** Chief Justice Wagner is a strong outlier in both interruptions made and received, consistent with the unique procedural role of the Chief Justice in managing oral hearings.


## 10. Limitations

- **Interruption detection is approximate.** We use timing heuristics rather than explicit markers (like the double-dash in US SCOTUS transcripts). This produces both false positives and false negatives.
- **Small sample of justices.** Only 10 justices served during this period (6 male, 4 female), limiting statistical power.
- **Chief Justice effect.** Wagner's procedural role distorts the "interruptions received" metric.
- **COVID-era hearings.** The 2020–2022 period included remote hearings, which may have altered interruption dynamics.
- **AI transcription errors.** Wallace's system occasionally misattributes speech during overlapping segments.


## 11. Save Data

In [ ]:
import json

# Save justice-level metrics
justice_df.to_csv(os.path.join(OUTPUT_DIR, "justice_metrics.csv"), index=False)

# Save case-level data
case_df.to_csv(os.path.join(OUTPUT_DIR, "case_level_metrics.csv"), index=False)

# Save interruptions data
with open(os.path.join(PROCESSED_DIR, "all_interruptions.json"), "w") as f:
    json.dump(all_interruptions, f, indent=2, ensure_ascii=False, default=str)

# Save time-to-first-interruption
if all_ttfi:
    pd.DataFrame(all_ttfi).to_csv(os.path.join(OUTPUT_DIR, "time_to_first_interruption.csv"), index=False)

# Save the report
with open(os.path.join(OUTPUT_DIR, "analysis_report.txt"), "w") as f:
    f.write(report)

print("All output saved to the output/ folder.")
